In [2]:
import tkinter as tk
from tkinter import scrolledtext
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import re
import random

# Load data from the given path
data_path = "C:\\Users\\Lenovo\\Downloads\\archive(2)\\intents.json"
with open(data_path, 'r') as f:
    data = json.load(f)

# Convert JSON data to DataFrame
df = pd.DataFrame(data['intents'])

# Create a Tkinter-based Chatbot GUI
class ChatbotGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Mental Health Chatbot")
        
        # Load and preprocess data
        self.preprocess_data()

        # Create GUI components
        self.create_widgets()

    def preprocess_data(self):
        dic = {"tag": [], "patterns": [], "responses": []}
        for example in data['intents']:
            for pattern in example['patterns']:
                dic['patterns'].append(pattern)
                dic['tag'].append(example['tag'])
                dic['responses'].append(example['responses'])

        self.df = pd.DataFrame.from_dict(dic)

        # Tokenization
        self.tokenizer = Tokenizer(lower=True, split=' ')
        self.tokenizer.fit_on_texts(self.df['patterns'])
        self.vocab_size = len(self.tokenizer.word_index)
        self.vocab_size = 1000

        # Encoding labels
        self.lbl_enc = LabelEncoder()
        self.y = self.lbl_enc.fit_transform(self.df['tag'])

        # Convert patterns to sequences and pad sequences
        ptrn2seq = self.tokenizer.texts_to_sequences(self.df['patterns'])
        self.X = pad_sequences(ptrn2seq, padding='post', maxlen=self.vocab_size + 1)
    def create_widgets(self):
        # Add GUI components
        self.user_input = tk.Entry(self.root, width=50)
        self.user_input.pack()

        self.conversation_display = scrolledtext.ScrolledText(self.root, width=60, height=20)
        self.conversation_display.pack()

        self.submit_button = tk.Button(self.root, text="Submit", command=self.get_user_input)
        self.submit_button.pack()

        # Initialize the chatbot model
        self.init_model()

    def init_model(self):
        self.model = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(self.vocab_size + 1,)),
            tf.keras.layers.Embedding(input_dim=self.vocab_size + 1, output_dim=100),
            tf.keras.layers.LSTM(32, return_sequences=True),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LSTM(32, return_sequences=True),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LSTM(32),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(len(np.unique(self.y)), activation="softmax")
        ])

        self.model.compile(optimizer='adam', loss="sparse_categorical_crossentropy", metrics=['accuracy'])

        self.model.summary()

        # Fit the model
        self.model.fit(x=self.X, y=self.y, batch_size=10,
                       callbacks=[tf.keras.callbacks.EarlyStopping(monitor='accuracy', patience=3)], epochs=100)

    def get_user_input(self):
        user_query = self.user_input.get()

        bot_response = self.generate_response(user_query)

        self.conversation_display.insert(tk.END, f"User: {user_query}\n")
        self.conversation_display.insert(tk.END, f"Chatbot: {bot_response}\n\n")

        self.user_input.delete(0, tk.END)

    def generate_response(self, user_input):
        text = []
        txt = re.sub('[^a-zA-Z\']', ' ', user_input)
        txt = txt.lower()
        txt = txt.split()
        txt = " ".join(txt)
        text.append(txt)

        x_test = self.tokenizer.texts_to_sequences(text)

        if not x_test:  # Check if x_test is empty or non-iterable
            return "Sorry, I couldn't understand that."

        x_test = pad_sequences(x_test, padding='post', maxlen=self.vocab_size + 1)  # Update padding length

        y_pred = self.model.predict(x_test)
        y_pred = y_pred.argmax()
        tag = self.lbl_enc.inverse_transform([y_pred])[0]
        responses = self.df[self.df['tag'] == tag]['responses'].values[0]

        return random.choice(responses)


# Create and run the GUI
root = tk.Tk()
chatbot_gui = ChatbotGUI(root)
root.mainloop()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 1001, 100)         100100    
                                                                 
 lstm_3 (LSTM)               (None, 1001, 32)          17024     
                                                                 
 layer_normalization_4 (Laye  (None, 1001, 32)         64        
 rNormalization)                                                 
                                                                 
 lstm_4 (LSTM)               (None, 1001, 32)          8320      
                                                                 
 layer_normalization_5 (Laye  (None, 1001, 32)         64        
 rNormalization)                                                 
                                                                 
 lstm_5 (LSTM)               (None, 32)               